In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("anime.csv")
print(df.head())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  


In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None


In [4]:
# Fill missing values
df['genre'] = df['genre'].fillna('')
df['type'] = df['type'].fillna('Unknown')
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [5]:
df.drop_duplicates(subset='name', inplace=True)

In [6]:
df['combined_features'] = df['genre'] + " " + df['type']

In [7]:
cv = CountVectorizer(stop_words='english')
count_matrix = cv.fit_transform(df['combined_features'])

In [8]:
scaler = MinMaxScaler()

df[['rating', 'episodes', 'members']] = scaler.fit_transform(
    df[['rating', 'episodes', 'members']]
)

In [9]:
cosine_sim = cosine_similarity(count_matrix)

In [10]:
def recommend_anime(anime_name, top_n=5, threshold=0.2):
    
    if anime_name not in df['name'].values:
        return "Anime not found!"

    idx = df[df['name'] == anime_name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove itself
    sim_scores = sim_scores[1:]
    
    # Apply threshold
    sim_scores = [i for i in sim_scores if i[1] > threshold]
    
    # Top N
    sim_scores = sim_scores[:top_n]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df['name'].iloc[anime_indices]

In [11]:
print(recommend_anime("Naruto", top_n=5, threshold=0.2))

841                     Naruto
206              Dragon Ball Z
515     Dragon Ball Kai (2014)
588            Dragon Ball Kai
1209       Medaka Box Abnormal
Name: name, dtype: object


User-Based:

Similar users ko find karta hai
"Agar A aur B similar users hai → B jo dekhta hai wo A ko recommend"

Item-Based:

Similar items (anime) ko find karta hai
"Agar tumne Naruto dekha → Naruto jaisa anime recommend hoga"

Definition:
Collaborative filtering ek technique hai jo users ke behavior (ratings, likes) ke basis pe recommendations deti hai.

Kaise kaam karta hai:

User data collect karta hai
Similarity find karta hai
Recommendation deta hai

 Example:

Tumne Attack on Titan dekha
Dusre log jinhe ye pasand aaya → unhone Death Note bhi dekha
System tumhe Death Note recommend karega